# 12 · Performance: plans, partitions, cost (Spark)

Each engine is fast in a different way, so this notebook has **different questions per engine**.

- **BigQuery** charges by **bytes read**. Columnar storage (each column stored apart): read fewer columns = pay less.
  **Partitioning** (split a table by date) and **clustering** (sort inside partitions) let it skip data.
- **Oracle** uses an **execution plan** made by the cost-based optimizer: full table scan, index, join method.
  `EXPLAIN PLAN` shows it.
- **Spark** builds a plan too (`explain()`). Look for **pushed filters**, **partition filters**,
  **broadcast joins** and **exchanges** (shuffles: data moving between machines, the slow part).

The tables here are tiny, so speed differences are small. Learn to **read** the plans: that is what interviews ask.

**How to run:** Start a cell with `%%sql` to write Spark SQL. In a normal Python cell use the DataFrame API: the tables are ready as variables (`employees`, `orders`, ...). Put your answer in `res` and call `res.show()`.

Tables: see [tables_guide.md](../../jarvis/tables_guide.md). Run the setup cell first (Shift+Enter).

> **Solutions.** Try the practice notebook first: [`spark/practice/12_performance.ipynb`](../../spark/practice/12_performance.ipynb)

In [ ]:
import sys, pathlib
ROOT = next(p for p in [pathlib.Path.cwd(), *pathlib.Path.cwd().parents]
            if (p / "settings.py").exists())   # the repo folder
sys.path.insert(0, str(ROOT / "spark"))
from spark_setup import get_spark, load_tables
from pyspark.sql import functions as F, Window, types as T

spark = get_spark()
t = load_tables(spark)          # also makes SQL views with the same names
employees, departments = t["employees"], t["departments"]
customers, orders = t["customers"], t["orders"]
products, order_items = t["products"], t["order_items"]
user_logins = t["user_logins"]
%load_ext spark_setup

## Q9. Read a Spark plan: pushed filters  ·  🟡 medium

Write `orders` to Parquet (a columnar file format) in `tmp/`, read it back, filter `status = 'completed'`, select 2 columns, and call `explain()`. Find **PushedFilters** and **ReadSchema** in the plan.

**How it works:** Write Parquet, read it back with a filter + select, and `explain()` shows the filter and columns pushed into the file scan.

In [ ]:
path = str(ROOT / "tmp" / "orders_parquet")
orders.write.mode("overwrite").parquet(path)
df = spark.read.parquet(path).filter(F.col("status") == "completed").select("order_id", "amount")
df.explain()

== Physical Plan ==
*(1) Project [order_id#9495, amount#9498]
+- *(1) Filter (isnotnull(status#9499) AND (status#9499 = completed))
   +- *(1) ColumnarToRow
      +- FileScan parquet [order_id#9495,amount#9498,status#9499] Batched: true, DataFilters: [isnotnull(status#9499), (status#9499 = completed)], Format: Parquet, Location: InMemoryFileIndex(1 paths)[file:<repo>/tmp/orders_parquet], PartitionFilters: [], PushedFilters: [IsNotNull(status), EqualTo(status,completed)], ReadSchema: struct<order_id:int,amount:decimal(10,2),status:string>

**Why / traps:** **Predicate pushdown**: the filter is sent to the file reader, so it skips data early. **Column pruning**: ReadSchema has only the columns you need. Both also happen with the BigQuery connector for Spark (that is why you select columns before reading).

## Q10. Broadcast join  ·  🟡 medium

Join `order_items` to `products` with a **broadcast** hint and `explain()` it. Find `BroadcastHashJoin`. Then do the same in Spark SQL with `/*+ BROADCAST(p) */`.

**How it works:** `F.broadcast(products)` (or the SQL hint) tells Spark to copy the small table to all executors; the plan shows BroadcastHashJoin.

In [ ]:
%%sql
SELECT /*+ BROADCAST(p) */ i.order_id, p.product_name, i.quantity
FROM order_items i
JOIN products p ON p.product_id = i.product_id
LIMIT 5

,order_id,product_name,quantity
0,1001,SQL Basics,1
1,1001,Statistics Made Easy,2
2,1001,Water Bottle,3
3,1002,Data Engineering 101,1
4,1003,Data Engineering 101,1


**DataFrame API:**

In [ ]:
order_items.join(F.broadcast(products), "product_id").explain()

== Physical Plan ==
AdaptiveSparkPlan isFinalPlan=false
+- Project [product_id#22, order_id#21, quantity#23, product_name#18, category#19, price#20]
   +- BroadcastHashJoin [product_id#22], [product_id#17], Inner, BuildRight, false
      :- Filter isnotnull(product_id#22)
      :  +- FileScan csv [order_id#21,product_id#22,quantity#23] Batched: false, DataFilters: [isnotnull(product_id#22)], Format: CSV, Location: InMemoryFileIndex(1 paths)[file:<repo>/data/order_items.tsv], PartitionFilters: [], PushedFilters: [IsNotNull(product_id)], ReadSchema: struct<order_id:int,product_id:int,quantity:int>
      +- BroadcastExchange HashedRelationBroadcastMode(List(cast(input[0, int, false] as bigint)),false), [plan_id=65680]
         +- Filter isnotnull(product_id#17)
            +- FileScan csv [product_id#17,product_name#18,category#19,price#20] Batched: false, DataFilters: [isnotnull(product_id#17)], Format: CSV, Location: InMemoryFileIndex(1 paths)[file:<repo>/data/products.tsv], PartitionFi

**Why / traps:** **Broadcast join**: the small table is copied to every executor, so the big table does **not** need a shuffle. Spark does it by itself under `spark.sql.autoBroadcastJoinThreshold` (10 MB). Without it you see `SortMergeJoin` with two `Exchange` steps.

## Q11. Partition pruning on files  ·  🟡 medium

Write `orders` as Parquet **partitioned by** `order_year` into `tmp/orders_by_year`. Read it back, filter `order_year = 2025`, explain. Find **PartitionFilters**.

**How it works:** partitionBy writes one folder per year; filtering on the year makes Spark read only that folder (PartitionFilters).

In [ ]:
path = str(ROOT / "tmp" / "orders_by_year")
(orders.withColumn("order_year", F.year("order_date"))
       .write.mode("overwrite").partitionBy("order_year").parquet(path))
df = spark.read.parquet(path).filter("order_year = 2025")
df.explain()
print(sorted(p.name for p in (ROOT / "tmp" / "orders_by_year").iterdir() if p.is_dir()))

== Physical Plan ==
*(1) ColumnarToRow
+- FileScan parquet [order_id#9527,customer_id#9528,order_date#9529,amount#9530,status#9531,order_year#9532] Batched: true, DataFilters: [], Format: Parquet, Location: InMemoryFileIndex(1 paths)[file:<repo>/tmp/orders_by_year], PartitionFilters: [isnotnull(order_year#9532), (order_year#9532 = 2025)], PushedFilters: [], ReadSchema: struct<order_id:int,customer_id:int,order_date:date,amount:decimal(10,2),status:string>


['order_year=2023', 'order_year=2024', 'order_year=2025']

**Why / traps:** `partitionBy` makes one folder per value (`order_year=2025/`). A filter on that column reads only the matching folders. Pick a column with **few** values (dates, not ids), or you get millions of tiny files.